# SIM Asset Intelligence: reproducible cost anomaly experiment

This notebook runs the repository's versioned training pipeline. It generates synthetic records by default, or accepts a CSV only when the data is authorized and de-identified.

**Required CSV columns:** `monthly_cost`, `expected_cost`. Optional evaluation label: `known_synthetic_anomaly` (0 = normal, 1 = anomaly).

A model flag is a review signal only. Do not upload employee names, phone numbers, ICCIDs, invoices, credentials or unapproved organizational data. Synthetic metrics are not evidence of real-world accuracy.

## 1. Prepare the runtime

The notebook clones the public repository so the same tested training script can be used locally and in Colab.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_DIR = Path('/content/sim-inventory-management-system')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/RURIGIMK/sim-inventory-management-system.git',
    str(REPO_DIR)
], check=True)
SCRIPT = REPO_DIR / 'notebooks' / 'sim_cost_model.py'
assert SCRIPT.is_file(), f'Training script not found: {SCRIPT}'
print('Repository and training script are ready.')

## 2. Configure the experiment

Keep `INPUT_CSV` as `None` to generate synthetic data. For an approved, de-identified CSV, set `UPLOAD_CSV = True` and upload the file when prompted. The dataset size is inferred from the CSV; it is not fixed to 600 records.

In [ ]:
UPLOAD_CSV = False
INPUT_CSV = None

RECORDS = 600              # Used only for synthetic generation
ANOMALY_RATE = 0.05       # Used only for synthetic generation
RANDOM_SEED = 42
TEST_SIZE = 0.30
N_ESTIMATORS = 250
CONTAMINATION = 0.05

if UPLOAD_CSV:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Upload exactly one CSV file.')
    INPUT_CSV = str(Path(next(iter(uploaded))).resolve())

MODEL_DIR = REPO_DIR / 'notebooks' / 'models'
RESULTS_DIR = REPO_DIR / 'notebooks' / 'results'
print('Input:', INPUT_CSV or 'generated synthetic dataset')
print('Model output:', MODEL_DIR)
print('Evaluation output:', RESULTS_DIR)

## 3. Train, evaluate and export

When labels are available, the pipeline holds out a stratified test set and compares Isolation Forest with a transparent cost-ratio baseline. When labels are absent, it fits the model but deliberately reports no precision, recall, F1 or false-positive rate.

In [ ]:
command = [
    sys.executable, str(SCRIPT),
    '--model-dir', str(MODEL_DIR),
    '--results-dir', str(RESULTS_DIR),
    '--records', str(RECORDS),
    '--anomaly-rate', str(ANOMALY_RATE),
    '--seed', str(RANDOM_SEED),
    '--test-size', str(TEST_SIZE),
    '--n-estimators', str(N_ESTIMATORS),
    '--contamination', str(CONTAMINATION),
]
if INPUT_CSV:
    command.extend(['--input-csv', str(INPUT_CSV)])

subprocess.run(command, cwd=REPO_DIR, check=True)

## 4. Review the generated outputs

The pipeline writes the model artifact and matching metadata to `notebooks/models/`, and the held-out evaluation CSV plus run summary to `notebooks/results/`.

In [ ]:
import json
import pandas as pd
from IPython.display import display

metadata = json.loads((MODEL_DIR / 'sim_cost_model_metadata.json').read_text())
print(json.dumps(metadata, indent=2))
evaluation = pd.read_csv(RESULTS_DIR / 'sim_anomaly_evaluation.csv')
display(evaluation.head(20))
print('Rows evaluated:', len(evaluation))

## 5. Download the outputs

The model, metadata and evaluation file are separate outputs. Keep the metadata with the model artifact so its feature order, training configuration, data fingerprint and evaluation limitations remain traceable.

In [ ]:
from google.colab import files
files.download(str(MODEL_DIR / 'sim_cost_isolation_forest.joblib'))
files.download(str(MODEL_DIR / 'sim_cost_model_metadata.json'))
files.download(str(RESULTS_DIR / 'sim_anomaly_evaluation.csv'))
files.download(str(RESULTS_DIR / 'run_summary.json'))